# Candidate Generator v2.6 — 16 kaçak için full-index rank tanısı

Bu notebook salt-okunur bir tanıdır. Mevcut Drive artifact'larını kullanır; dataset ZIP'ini açmaz, görüntüleri göstermez, duplicate scoring/family/split/model training çalıştırmaz.

Hücreleri yukarıdan aşağıya çalıştır. Sonuç görülmeden V2.6 kuralı seçilmeyecektir.

In [ ]:
from google.colab import drive
from pathlib import Path
import datetime, json, os, subprocess, sys

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Git commit:', commit)

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
assert torch.cuda.is_available(), 'T4 GPU runtime seçilmemiş.'


In [ ]:
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_3_full_retrieval_v1')
REPORT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_6_missed_pair_rank_diagnostic_v1.json')
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOG = Path(f'/content/drive/MyDrive/NNGA_OUTPUTS/candidate_generator_v2_6_missed_pair_rank_diagnostic_{timestamp}.log')
CONFIG = PROJECT_ROOT / 'config' / 'candidate_generator_v2_6_missed_pair_rank_diagnostic.json'
ANALYZER = PROJECT_ROOT / 'src' / 'analyze_candidate_generator_v2_6_missed_pair_ranks.py'

assert (OUTPUT_DIR / 'stage_manifest.json').is_file(), '1.107-artifact stage manifest bulunamadı.'
assert (OUTPUT_DIR / 'run_contract.json').is_file(), 'Run contract bulunamadı.'
assert CONFIG.is_file(), 'Dondurulmuş tanı config dosyası bulunamadı.'
assert CONFIG.with_suffix('.sha256').is_file(), 'Config SHA-256 dosyası bulunamadı.'
assert ANALYZER.is_file(), 'Rank analyzer bulunamadı.'
manifest = json.loads((OUTPUT_DIR / 'stage_manifest.json').read_text(encoding='utf-8'))
print('Kayıtlı artifact:', len(manifest['artifacts']))
assert len(manifest['artifacts']) == 1107, 'Artifact sayısı 1.107 değil.'
print('Ön kontrol başarılı. Dataset ZIP veya görüntü klasörü gerekmiyor.')
print('Rapor:', REPORT)
print('Log:', LOG)


In [ ]:
command = [
    sys.executable, '-u', str(ANALYZER),
    '--output-dir', str(OUTPUT_DIR),
    '--report', str(REPORT),
    '--config', str(CONFIG),
    '--block-size', '256',
    '--device', 'cuda',
]
print('Tanı başlıyor. Komut:', ' '.join(command))
print('Not: üç source group ve dört descriptor kanalı işlendiği için ilerleme satırları birkaç dakika sürebilir.')
with LOG.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Rank tanısı tamamlanamadı. Drive artifactları değiştirilmedi; logu Codex’e gönder.')

report = json.loads(REPORT.read_text(encoding='utf-8'))
print('\nDurum:', report['status'])
print('Doğrulanan artifact:', report['input']['verified_artifacts_in_this_diagnostic'])
for group, result in report['by_source_group'].items():
    print(
        group,
        '| pair:', result['target_pair_count'],
        '| minimum observed K:', result['minimum_k_covering_all_observed_targets'],
        '| first grid K:', result['first_predeclared_grid_k_covering_all'],
    )
profile = report['diagnostic_only_first_grid_profile']
print('Tanısal ilk-grid profil:', profile['top_k_by_group'])
print('Projected full candidates:', profile['full_candidate_pairs'])
print('Pairs/image:', profile['full_pairs_per_image'])
print('Capacity <=25:', profile['passes_25_pairs_per_image_capacity'])
print('Bu sonuç henüz V2.6 kuralı değildir; JSON dosyasını Codex’e gönder.')


In [ ]:
from google.colab import files
assert REPORT.is_file(), 'Rapor henüz oluşmadı.'
files.download(str(REPORT))
